In [23]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import sklearn
import torch
from torchvision import transforms
import torchmetrics
#from google.colab import ai

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/dataset.csv"
local_path = "../FracAtlas/dataset.csv"
df = pl.read_csv(colab_path)
df.head()


image_id,hand,leg,hip,shoulder,mixed,hardware,multiscan,fractured,fracture_count,frontal,lateral,oblique
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""IMG0000000.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000001.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000002.jpg""",0,1,0,0,0,0,1,0,0,1,1,0
"""IMG0000003.jpg""",0,1,0,0,0,0,1,0,0,0,1,1
"""IMG0000004.jpg""",0,1,0,0,0,0,1,0,0,0,1,1


In [26]:
fractured = df.filter(df["fractured"] == 1)
nonfractured = df.filter(df["fractured"] == 0)

train, test_validate = sklearn.model_selection.train_test_split(df, stratify=df["fractured"], train_size=0.7, test_size=0.3, random_state=42)
test, validation = sklearn.model_selection.train_test_split(test_validate, stratify=test_validate["fractured"], train_size=0.5, test_size=0.5, random_state=42)

fractured_local_path = "../FracAtlas/images/Fractured/"
non_fractured_local_path = "../FracAtlas/images/Non_fractured/"

fractured_colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Fractured/"
non_fractured_colab_path = "/content/drive/MyDrive/STINTSYMCO1/FracAtlas/images/Non_fractured/"

train_paths = train.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_colab_path + pl.col("image_id")).otherwise(non_fractured_colab_path + pl.col("image_id")).alias("paths")
)

test_paths = test.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_colab_path + pl.col("image_id")).otherwise(non_fractured_colab_path + pl.col("image_id")).alias("paths")
)

validation_paths = validation.with_columns(
    pl.when(pl.col("fractured") == 1).then(fractured_colab_path + pl.col("image_id")).otherwise(non_fractured_colab_path + pl.col("image_id")).alias("paths")
)

train_paths = train_paths["paths"]
test_paths = test_paths["paths"]
validation_paths = validation_paths["paths"]

train_labels = train["fractured"]
test_labels = test["fractured"]
validation_labels = validation["fractured"]




In [27]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                          
    transforms.Normalize([0.485, 0.456, 0.406],    
                         [0.229, 0.224, 0.225])   
])


In [28]:
from torch.utils.data import Dataset 
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

class FractureDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self,index):
        img = Image.open(self.paths[index]).convert("RGB")
        img = self.transform(img)
        label = self.labels[index]
        return img, label

In [29]:
from torch.utils.data import DataLoader

use_cuda = torch.cuda.is_available()
num_workers = 2 if use_cuda else 0
pin_memory = use_cuda

train_dataset = FractureDataset(train_paths.to_list(), train_labels.to_list(), transform=transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory
)

val_dataset = FractureDataset(validation_paths.to_list(), validation_labels.to_list(), transform)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

test_dataset = FractureDataset(test_paths.to_list(), test_labels.to_list(), transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

print(f"CUDA available: {use_cuda} | num_workers={num_workers} | pin_memory={pin_memory}")

CUDA available: True | num_workers=2 | pin_memory=True


In [30]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 1) 
)

print("Trainable params:", sum(p.requires_grad for p in model.parameters()))

Trainable params: 2


In [31]:
num_fractured = int((train["fractured"] == 1).sum())
num_non_fractured = int((train["fractured"] == 0).sum())
print(f"Train set class counts -> fractured: {num_fractured}, non-fractured: {num_non_fractured}")

Train set class counts -> fractured: 502, non-fractured: 2356


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

pos_weight = torch.tensor([num_non_fractured / num_fractured], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

print(f"Using device: {device} | pos_weight={pos_weight.item():.4f}")

Using device: cuda | pos_weight=4.6932


In [ ]:
model = model.to(device)
model_device = next(model.parameters()).device
print(f"Model device: {model_device}")

sample_images, sample_labels = next(iter(train_loader))
print(f"Batch before move -> images: {sample_images.device}, labels: {sample_labels.device}")
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)
print(f"Batch after move  -> images: {sample_images.device}, labels: {sample_labels.device}")

assert model_device.type == device.type, "Model is not on the expected device."
assert sample_images.device.type == device.type, "Input batch is not on the expected device."

Model device: cuda:0
Batch before move -> images: cpu, labels: cpu
Batch after move  -> images: cuda:0, labels: cuda:0


Base Model

In [ ]:
num_epochs = 10
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score
from torch.cuda.amp import GradScaler, autocast
accuracy = BinaryAccuracy().to(device)
precision = BinaryPrecision().to(device)
recall = BinaryRecall().to(device)
f1 = BinaryF1Score().to(device)

scaler = GradScaler()

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    accuracy.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        labels_float = labels.float()
        labels_int = labels.long()

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images).flatten()
            loss = criterion(outputs, labels_float)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)  
        scaler.update()
        


        train_loss += loss.item() * images.size(0)
        preds = (outputs > 0).long()

        accuracy.update(preds, labels_int)
        precision.update(preds, labels_int)
        recall.update(preds, labels_int)
        f1.update(preds, labels_int)

    train_loss /= len(train_dataset)
    train_acc = accuracy.compute().item()
    train_precision = precision.compute().item()
    train_recall = recall.compute().item()
    train_f1 = f1.compute().item()

    model.eval()
    val_loss = 0.0

    accuracy.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            labels_float = labels.float()
            labels_int = labels.long()

            outputs = model(images).flatten()
            loss = criterion(outputs, labels_float)

            val_loss += loss.item() * images.size(0)
            preds = (outputs > 0).long()

            accuracy.update(preds, labels_int)
            precision.update(preds, labels_int)
            recall.update(preds, labels_int)
            f1.update(preds, labels_int)

    val_loss /= len(val_dataset)
    val_acc = accuracy.compute().item()
    val_precision = precision.compute().item()
    val_recall = recall.compute().item()
    val_f1 = f1.compute().item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} Prec: {train_precision:.4f} Rec: {train_recall:.4f} F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Prec: {val_precision:.4f} Rec: {val_recall:.4f} F1: {val_f1:.4f}"
    )

/tmp/ipykernel_32503/1966128731.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch [1/10] Train Loss: 1.0343 Acc: 0.6487 Prec: 0.2821 Rec: 0.6474 F1: 0.3930 | Val Loss: 0.8956 Acc: 0.7259 Prec: 0.3661 Rec: 0.7593 F1: 0.4940
Epoch [2/10] Train Loss: 0.8786 Acc: 0.7453 Prec: 0.3813 Rec: 0.7231 F1: 0.4993 | Val Loss: 0.8581 Acc: 0.6868 Prec: 0.3359 Rec: 0.7963 F1: 0.4725
Epoch [3/10] Train Loss: 0.8239 Acc: 0.7617 Prec: 0.4037 Rec: 0.7470 F1: 0.5241 | Val Loss: 0.8527 Acc: 0.8157 Prec: 0.4825 Rec: 0.6389 F1: 0.5498
Epoch [4/10] Train Loss: 0.8283 Acc: 0.7579 Prec: 0.4006 Rec: 0.7629 F1: 0.5254 | Val Loss: 0.8212 Acc: 0.8042 Prec: 0.4605 Rec: 0.6481 F1: 0.5385
Epoch [5/10] Train Loss: 0.7808 Acc: 0.7799 Prec: 0.4291 Rec: 0.7649 F1: 0.5497 | Val Loss: 0.8207 Acc: 0.8157 Prec: 0.4825 Rec: 0.6389 F1: 0.5498
Epoch [6/10] Train Loss: 0.7746 Acc: 0.7729 Prec: 0.4197 Rec: 0.7649 F1: 0.5420 | Val Loss: 0.8418 Acc: 0.8320 Prec: 0.5191 Rec: 0.6296 F1: 0.5690
Epoch [7/10] Train Loss: 0.7742 Acc: 0.7733 Prec: 0.4201 Rec: 0.7649 F1: 0.5424 | Val Loss: 0.8293 Acc: 0.8287 Prec: 0

In [ ]:

save_path = "/content/drive/MyDrive/STINTSYMCO1/finetunedNN.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved to: {save_path}")


Saved to: /content/drive/MyDrive/STINTSYMCO1/finetunedNN.pth
Checkpoint saved to: /content/drive/MyDrive/STINTSYMCO1/checkpoint.pth


In [37]:

for p in model.features[-2].parameters():
    p.requires_grad = True
for p in model.features[-1].parameters():
    p.requires_grad = True
    
optimizer = torch.optim.Adam([
    {"params": model.classifier.parameters(), "lr": 1e-3},
    {"params": model.features[-1].parameters(), "lr": 1e-4},
    {"params": model.features[-2].parameters(), "lr": 1e-5},
])
print("Trainable params:", sum(p.requires_grad for p in model.parameters()))

Trainable params: 14


# We unfreeze penultimate layer to give the model more variance.

In [35]:
torch.amp.autocast_mode.is_autocast_available(device.type)


True

In [38]:
num_epochs = 10
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score
from torch.cuda.amp import GradScaler, autocast
accuracy = BinaryAccuracy().to(device)
precision = BinaryPrecision().to(device)
recall = BinaryRecall().to(device)
f1 = BinaryF1Score().to(device)

scaler = GradScaler()

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    accuracy.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        labels_float = labels.float()
        labels_int = labels.long()

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images).flatten()
            loss = criterion(outputs, labels_float)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)  
        scaler.update()
        


        train_loss += loss.item() * images.size(0)
        preds = (outputs > 0).long()

        accuracy.update(preds, labels_int)
        precision.update(preds, labels_int)
        recall.update(preds, labels_int)
        f1.update(preds, labels_int)

    train_loss /= len(train_dataset)
    train_acc = accuracy.compute().item()
    train_precision = precision.compute().item()
    train_recall = recall.compute().item()
    train_f1 = f1.compute().item()

    model.eval()
    val_loss = 0.0

    accuracy.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            labels_float = labels.float()
            labels_int = labels.long()

            outputs = model(images).flatten()
            loss = criterion(outputs, labels_float)

            val_loss += loss.item() * images.size(0)
            preds = (outputs > 0).long()

            accuracy.update(preds, labels_int)
            precision.update(preds, labels_int)
            recall.update(preds, labels_int)
            f1.update(preds, labels_int)

    val_loss /= len(val_dataset)
    val_acc = accuracy.compute().item()
    val_precision = precision.compute().item()
    val_recall = recall.compute().item()
    val_f1 = f1.compute().item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} Prec: {train_precision:.4f} Rec: {train_recall:.4f} F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} Prec: {val_precision:.4f} Rec: {val_recall:.4f} F1: {val_f1:.4f}"
    )

/tmp/ipykernel_43233/746783851.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


/tmp/ipykernel_43233/746783851.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch [1/10] Train Loss: 0.1939 Acc: 0.9566 Prec: 0.8225 Rec: 0.9602 F1: 0.8860 | Val Loss: 1.2774 Acc: 0.8434 Prec: 0.5429 Rec: 0.7037 F1: 0.6129
Epoch [2/10] Train Loss: 0.1413 Acc: 0.9675 Prec: 0.8569 Rec: 0.9781 F1: 0.9135 | Val Loss: 1.5463 Acc: 0.8597 Prec: 0.5948 Rec: 0.6389 F1: 0.6161
Epoch [3/10] Train Loss: 0.1538 Acc: 0.9636 Prec: 0.8443 Rec: 0.9721 F1: 0.9037 | Val Loss: 1.6360 Acc: 0.8581 Prec: 0.5897 Rec: 0.6389 F1: 0.6133
Epoch [4/10] Train Loss: 0.1207 Acc: 0.9713 Prec: 0.8723 Rec: 0.9801 F1: 0.9231 | Val Loss: 1.8749 Acc: 0.8630 Prec: 0.6111 Rec: 0.6111 F1: 0.6111
Epoch [5/10] Train Loss: 0.1124 Acc: 0.9738 Prec: 0.8806 Rec: 0.9841 F1: 0.9294 | Val Loss: 1.7914 Acc: 0.8646 Prec: 0.6087 Rec: 0.6481 F1: 0.6278
Epoch [6/10] Train Loss: 0.1064 Acc: 0.9762 Prec: 0.8903 Rec: 0.9861 F1: 0.9357 | Val Loss: 1.8858 Acc: 0.8548 Prec: 0.5841 Rec: 0.6111 F1: 0.5973
Epoch [7/10] Train Loss: 0.1149 Acc: 0.9717 Prec: 0.8834 Rec: 0.9661 F1: 0.9229 | Val Loss: 1.8931 Acc: 0.8467 Prec: 0

In [ ]:

save_path = "/content/drive/MyDrive/STINTSYMCO1/finetunedNN_2.0.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved to: {save_path}")

In [ ]:
torch.save(model.state_dict(), 'model.pth')
print('Model saved!')